# imports

In [2]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from neuralforecast import NeuralForecast
import optuna
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import pandas as pd
from pathlib import Path
import pathlib
import math
import holidays
import json
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import os
import random
import torch
import warnings
import xgboost as xgb
import lightgbm as lgb
import torch
torch.set_float32_matmul_precision("medium")
from neuralforecast.models import NHITS
from sklearn.metrics import root_mean_squared_error
from neuralforecast.losses.pytorch import MSE


In [3]:

# ============================================================
# GLOBAL SETTINGS FOR LIGHTWEIGHT HPO
# ============================================================
MAX_STEPS = 50
VAL_CHECK_STEPS = MAX_STEPS // 10


def split_train_val_test_global(df_nf, test_start, forecast_horizon, training_size, val_days=3, freq_minutes=15):
    """
    Split a long NeuralForecast dataframe into train / validation / test per series.

    Validation starts `val_days` before test_start and ends right before test_start.
    Test spans `forecast_horizon` steps starting at test_start.
    Training is the last `training_size` rows before validation starts.
    """
    test_start = pd.Timestamp(test_start)
    val_start = test_start - pd.Timedelta(days=val_days)
    test_end = test_start + pd.Timedelta(minutes=freq_minutes * forecast_horizon)

    train_parts = []
    val_parts = []
    test_parts = []

    expected_val_len = val_days * (24 * 60 // freq_minutes)

    for uid, g in df_nf.groupby("unique_id"):
        g = g.sort_values("ds").reset_index(drop=True)

        # train: everything before validation starts, keep only last training_size rows
        train_candidates = g[g["ds"] < val_start].copy()
        train_df_uid = train_candidates.iloc[-training_size:].copy()

        # validation: from val_start until just before test_start
        val_df_uid = g[(g["ds"] >= val_start) & (g["ds"] < test_start)].copy()

        # test: from test_start for forecast_horizon steps
        test_df_uid = g[(g["ds"] >= test_start) & (g["ds"] < test_end)].copy()

        # safety checks
        if len(train_df_uid) != training_size:
            raise ValueError(f"{uid}: expected {training_size} training rows, got {len(train_df_uid)}")

        if len(val_df_uid) != expected_val_len:
            raise ValueError(f"{uid}: expected {expected_val_len} validation rows, got {len(val_df_uid)}")

        if len(test_df_uid) != forecast_horizon:
            raise ValueError(f"{uid}: expected {forecast_horizon} test rows, got {len(test_df_uid)}")

        train_parts.append(train_df_uid)
        val_parts.append(val_df_uid)
        test_parts.append(test_df_uid)

    train_df = pd.concat(train_parts, ignore_index=True)
    val_df = pd.concat(val_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)

    return train_df, val_df, test_df


def build_global_nf_df(df_all, home_cols, weather_cols=None):
    """
    Convert a wide household load dataframe into NeuralForecast long format.

    Parameters
    ----------
    df_all : pd.DataFrame
        Index must be DatetimeIndex, columns include home_* and optional weather cols.
    home_cols : list
        List of household columns, e.g. ['home_1', 'home_2', ...]
    weather_cols : list or None
        Optional list of weather columns to attach to every home/timestamp row.

    Returns
    -------
    df_nf : pd.DataFrame
        Columns: unique_id, ds, y, [weather columns...]
    """
    if not isinstance(df_all.index, pd.DatetimeIndex):
        raise ValueError("df_all index must be a DatetimeIndex.")

    # keep time as a normal column
    df_base = df_all.reset_index().rename(columns={"timestamp": "ds"})

    # wide -> long for homes
    df_nf = df_base.melt(
        id_vars=["ds"],
        value_vars=home_cols,
        var_name="unique_id",
        value_name="y"
    )

    # add weather columns if requested
    if weather_cols is not None and len(weather_cols) > 0:
        weather_df = df_base[["ds"] + weather_cols].copy()
        df_nf = df_nf.merge(weather_df, on="ds", how="left")

    # sort for safety
    df_nf = df_nf.sort_values(["unique_id", "ds"]).reset_index(drop=True)

    return df_nf

def build_daily_profile_matrix(df_all, home_cols):
    df_tmp = df_all.copy()
    df_tmp["slot"] = df_tmp.index.hour * 4 + df_tmp.index.minute // 15

    profiles = []
    for home in home_cols:
        avg_profile = df_tmp.groupby("slot")[home].mean()
        avg_profile.name = home
        profiles.append(avg_profile)

    profile_df = pd.concat(profiles, axis=1).T
    profile_df.index.name = "home"

    return profile_df






def rolling_forecasting_validation_predictions(train_df, val_df, h, model_params, freq="15min"):
    rolling_train_df = train_df.copy()
    val_predictions = []

    val_starts = sorted(val_df["ds"].unique())[::h]

    for window_start in val_starts:

        model = NHITS(
            h=h,
            input_size=model_params["input_size"],
            hist_exog_list=model_params["hist_exog_list"],
            stack_types=model_params["stack_types"],
            n_blocks=model_params["n_blocks"],
            mlp_units=model_params["mlp_units"],
            n_pool_kernel_size=model_params["n_pool_kernel_size"],
            n_freq_downsample=model_params["n_freq_downsample"],
            pooling_mode=model_params["pooling_mode"],
            interpolation_mode=model_params["interpolation_mode"],
            dropout_prob_theta=model_params["dropout_prob_theta"],
            activation=model_params["activation"],
            batch_size=model_params["batch_size"],
            learning_rate=model_params["learning_rate"],
            max_steps=MAX_STEPS,
            val_check_steps=min(VAL_CHECK_STEPS, MAX_STEPS),
            random_seed=42,
            loss=MSE(),
        )

        nf = NeuralForecast(models=[model], freq=freq)
        nf.fit(df=rolling_train_df)

        preds = nf.predict()
        val_predictions.append(preds)

        next_val_chunk = val_df[
            (val_df["ds"] >= window_start) &
            (val_df["ds"] < window_start + pd.Timedelta(minutes=15 * h))
        ].copy()

        rolling_train_df = pd.concat([rolling_train_df, next_val_chunk], ignore_index=True)
        rolling_train_df = rolling_train_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

    return pd.concat(val_predictions, ignore_index=True)




def compute_average_rmse_per_cluster(val_df, val_preds_df, pred_col="NHITS"):
    val_compare_df = val_df.merge(val_preds_df, on=["unique_id", "ds"], how="left")

    rmse_rows = []
    for uid, g in val_compare_df.groupby("unique_id"):
        rmse_rows.append({
            "unique_id": uid,
            "RMSE": root_mean_squared_error(g["y"], g[pred_col])
        })

    rmse_per_home = pd.DataFrame(rmse_rows).sort_values("RMSE").reset_index(drop=True)
    avg_rmse_cluster = rmse_per_home["RMSE"].mean()

    return avg_rmse_cluster, rmse_per_home, val_compare_df


def objective(trial):

    model_params = {
        "input_size": trial.suggest_categorical("input_size", [
            forecast_horizon * 3,
            forecast_horizon * 5
        ]),
        "stack_types": ["identity", "identity", "identity"],
        "n_blocks": trial.suggest_categorical("n_blocks", [
            [1, 1, 1],
            [2, 2, 2]
        ]),
        "mlp_units": trial.suggest_categorical("mlp_units", [
            3 * [[256, 256]],
            3 * [[512, 512]]
        ]),
        "n_pool_kernel_size": trial.suggest_categorical("n_pool_kernel_size", [
            [2, 2, 1],
            [4, 2, 1]
        ]),
        "n_freq_downsample": trial.suggest_categorical("n_freq_downsample", [
            [4, 2, 1],
            [8, 4, 1]
        ]),
        "pooling_mode": trial.suggest_categorical("pooling_mode", [
            "MaxPool1d",
            "AvgPool1d"
        ]),
        "interpolation_mode": "linear",
        "dropout_prob_theta": trial.suggest_float("dropout_prob_theta", 0.0, 0.1),
        "activation": "ReLU",
        "batch_size": trial.suggest_categorical("batch_size", [32, 64]),
        "learning_rate": trial.suggest_float("learning_rate", 5e-4, 2e-3, log=True),
        "hist_exog_list": weather_cols,
    }

    try:
        val_preds_df = rolling_forecasting_validation_predictions(
            train_df=train_df,
            val_df=val_df,
            h=forecast_horizon,
            model_params=model_params,
            freq="15min"
        )

        avg_rmse_cluster, _, _ = compute_average_rmse_per_cluster(
            val_df=val_df,
            val_preds_df=val_preds_df,
            pred_col="NHITS"
        )

        return avg_rmse_cluster

    except Exception as e:
        print(f"Trial failed: {e}")
        return float("inf")

# start

In [4]:

project_path = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone"
days_json_path = pathlib.Path(project_path) / "dataset_days.json"

#countries = ["Germany", "Ireland", "Portugal"]
countries = ["Germany"]

#days = ["day1", "day2", "day3", "day4", "day5"]
days = ["day1"]


forecast_horizon = 96
training_size = 96 * 7 * 3 * 2
feature_selection = True
plot_forecast = True
hyperparameter_opt = True
opt_trials = 2

weather_cols = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation"
]

# -------------------------
# Read JSON with forecast days
# -------------------------
with open(days_json_path, "r") as f:
    dataset_days = json.load(f)

# -------------------------
# Loop over countries
# -------------------------
for country in countries:
    print(f"\n{'#'*100}")
    print(f"COUNTRY: {country}")
    print(f"{'#'*100}")

    dataset_path = pathlib.Path(project_path) / "DataCleaning" / "clean" / f"dataset_{country}.csv"

    df_all = pd.read_csv(dataset_path, parse_dates=["timestamp"])
    df_all = df_all.set_index("timestamp")
    df_all = df_all.sort_index()

    home_cols = [col for col in df_all.columns if col.startswith("home_")]

    print(f"Detected {len(home_cols)} homes for {country}.")
    print(home_cols)

    # make the dataset of each country clustered
    df_nf = build_global_nf_df(df_all, home_cols, weather_cols)
    profile_df = build_daily_profile_matrix(df_all, home_cols)

    print(profile_df.head())
    print(profile_df.shape)   # should be (28, 96)

    scaler = StandardScaler()
    X_profile = scaler.fit_transform(profile_df)

    results = []
    for k in range(2, 6):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels = kmeans.fit_predict(X_profile)
        score = silhouette_score(X_profile, labels)

        results.append({"k": k, "silhouette_score": score})

    results_df = pd.DataFrame(results).sort_values("silhouette_score", ascending=False)
    print(results_df)

    best_k = int(results_df.iloc[0]["k"])
    best_score = results_df.iloc[0]["silhouette_score"]

    print(f"Best k: {best_k}")
    print(f"Best silhouette score: {best_score:.4f}")

    best_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
    best_labels = best_kmeans.fit_predict(X_profile)

    cluster_profile_df = profile_df.copy()
    cluster_profile_df["cluster"] = best_labels

    print(cluster_profile_df["cluster"].sort_values())
    # finish clustering
    
    # -------------------------
    # Loop over days
    # -------------------------
    for day_name in days:
        selected_day = dataset_days[country][day_name]
        date = f"{selected_day} 00:00:00"
        forecast_end_date = str(pd.Timestamp(date) + pd.Timedelta(days=1))

        print(f"\n{'='*100}")
        print(f"Running {country} - {day_name}")
        print(f"Forecast start: {date}")
        print(f"Forecast end:   {forecast_end_date}")
        print(f"{'='*100}")


        clusters=best_k
        cluster_test_preds_list = [] # for the predictions
        for cluster in range(clusters):
            print(cluster)

            # --------------------------------------------------
            # iterate over clusters
            # --------------------------------------------------
            selected_cluster = cluster   # change to 1 if you want the other one later

            cluster_homes = cluster_profile_df.index[cluster_profile_df["cluster"] == selected_cluster].tolist()

            print(f"Selected cluster: {selected_cluster}")
            print(f"Number of homes in cluster: {len(cluster_homes)}")
            print("Homes in cluster:")
            print(cluster_homes)


            # --------------------------------------------------
            # subset original dataframe to homes in this cluster
            # --------------------------------------------------
            cluster_cols = cluster_homes + weather_cols
            df_cluster_wide = df_all[cluster_cols].copy()

            print("\nCluster-wide dataframe head:")
            print(df_cluster_wide.head())


            df_cluster_nf = build_global_nf_df(
                df_all=df_cluster_wide,
                home_cols=cluster_homes,
                weather_cols=weather_cols
            )

            print("\nCluster long-format dataset:")
            print(df_cluster_nf.head(10))

            print("\nColumns:")
            print(df_cluster_nf.columns.tolist())

            print("\nShape:")
            print(df_cluster_nf.shape)

            print("\nUnique homes in long dataset:")
            print(df_cluster_nf['unique_id'].unique())

            train_df, val_df, test_df = split_train_val_test_global(
                df_nf=df_cluster_nf,
                test_start=date,
                forecast_horizon=forecast_horizon,
                training_size=training_size,
                val_days=3,
                freq_minutes=15
            )

            for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
                start = df['ds'].min()
                end = df['ds'].max()
                print(f"{name}: {start} to {end} (Shape: {df.shape})")

            study = optuna.create_study(direction="minimize")
            study.optimize(objective, n_trials=opt_trials, show_progress_bar=True)

            print("Best avg RMSE:", study.best_value)
            print("Best params:", study.best_params)


            # now we keep the best parameters and we predict the test
            best_params = study.best_params

            train_val_df = pd.concat([train_df, val_df], ignore_index=True)
            train_val_df = train_val_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)


            final_nhits = NHITS(
                h=forecast_horizon,
                input_size=best_params["input_size"],
                hist_exog_list=weather_cols,
                stack_types=["identity", "identity", "identity"],
                n_blocks=best_params["n_blocks"],
                mlp_units=best_params["mlp_units"],
                n_pool_kernel_size=best_params["n_pool_kernel_size"],
                n_freq_downsample=best_params["n_freq_downsample"],
                pooling_mode=best_params["pooling_mode"],
                interpolation_mode="linear",
                dropout_prob_theta=best_params["dropout_prob_theta"],
                activation="ReLU",
                batch_size=best_params["batch_size"],
                learning_rate=best_params["learning_rate"],
                max_steps=MAX_STEPS,
                val_check_steps=min(VAL_CHECK_STEPS, MAX_STEPS),
                random_seed=42,
                loss=MSE()
            )

            nf_final = NeuralForecast(
                models=[final_nhits],
                freq="15min",
            )

            nf_final.fit(df=train_val_df)

            test_preds_df = nf_final.predict()

            test_preds_wide = test_preds_df.pivot(
                index="ds",
                columns="unique_id",
                values="NHITS"
            ).sort_index()
            cluster_test_preds_list.append(test_preds_wide)
        final_test_preds_wide = pd.concat(cluster_test_preds_list, axis=1).sort_index()


        # --------------------------------------------------
        # save final combined test predictions
        # --------------------------------------------------
        save_dir = pathlib.Path(project_path) / "Outputs" / "Global models" / "NHITS"
        save_dir.mkdir(parents=True, exist_ok=True)

        save_path = save_dir / f"prediction_NHITS_{day_name}_{country}.csv"

        final_test_preds_wide.to_csv(save_path, index=True)

        print(f"Saved final_test_preds_wide to: {save_path}")




####################################################################################################
COUNTRY: Germany
####################################################################################################
Detected 28 homes for Germany.
['home_1', 'home_2', 'home_3', 'home_4', 'home_5', 'home_6', 'home_7', 'home_8', 'home_9', 'home_10', 'home_11', 'home_12', 'home_13', 'home_14', 'home_15', 'home_16', 'home_17', 'home_18', 'home_19', 'home_20', 'home_21', 'home_22', 'home_23', 'home_24', 'home_25', 'home_26', 'home_27', 'home_28']
slot             0            1            2            3            4   \
home                                                                      
home_1   252.102996   261.048660   246.110175   257.327671   454.420217   
home_2   934.975708   902.009367   866.633501   901.735967   867.579963   
home_3  1066.707239  1058.940145   990.620198   993.305127   990.090731   
home_4   550.102459   525.683797   537.442987   514.869658   524.066859   

[I 2026-03-19 15:59:56,004] A new study created in memory with name: no-name-db78967c-b8f9-440f-9116-b1788b7edc26


Train: 2019-11-14 00:00:00 to 2019-12-25 23:45:00 (Shape: (68544, 8))
Val: 2019-12-26 00:00:00 to 2019-12-28 23:45:00 (Shape: (4896, 8))
Test: 2019-12-29 00:00:00 to 2019-12-29 23:45:00 (Shape: (1632, 8))


  0%|          | 0/2 [00:00<?, ?it/s]

c:\Users\CR58XM\AppData\Local\anaconda3\envs\sktime\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [1, 1, 1] which is of type list.
  optuna_warn(message)
c:\Users\CR58XM\AppData\Local\anaconda3\envs\sktime\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [2, 2, 2] which is of type list.
  optuna_warn(message)
c:\Users\CR58XM\AppData\Local\anaconda3\envs\sktime\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [[256, 256], [256, 256], [256, 256]] which is of type list.
  optuna_warn(message)
c:\Users\CR58XM\AppData\Local\anaconda3\envs\sktime\Lib\site-packages\optuna\distributio

...

Output()

`Trainer.fit` stopped: `max_steps=50` reached.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

Saved final_test_preds_wide to: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Global models\NHITS\prediction_NHITS_day1_Germany.csv


# end 

it takes around 3 hours